# 音声で話せる対話システムを作る（音声認識・LLM・音声合成）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_dialogue/voice_dialogue.ipynb)

NLP クックブック 26-14 ／ 対話システム

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

マイクに話しかけると声で答えるボットを作りたい。音声認識 (Whisper) で書き起こし、LLM で応答を作り、音声合成 (Open JTalk) で読み上げる 3 段をつなぎ、話し終えてから返事が聞こえるまでの遅延がどこで生じるかを測る。

In [1]:
# pyopenjtalk は PyPI にソースだけがあり、インストール時にビルドする（Colab で 2〜5 分ほど）
!pip install -q pyopenjtalk==0.4.1

In [2]:
import numpy as np
import pyopenjtalk
from IPython.display import Audio, display
from scipy.signal import resample_poly

SR = 16000                                     # Whisper は 16kHz のモノラルを前提にする

def tts(text):
    """Open JTalk で読み上げ、16kHz の float32 の波形にする。初回の呼び出しで辞書（約 23MB）をダウンロードする"""
    wav, sr = pyopenjtalk.tts(text)            # 48kHz。値は int16 の範囲の float
    return resample_poly(wav / 32768, SR, sr).astype(np.float32)

# 利用者の発話の代わりに、音声合成で作った音声を入力にする（マイクの録音を使う方法は注を参照）
user_texts = ["こんにちは。今度の週末に京都へ行くんですが、雨の日でも楽しめる場所はありますか？",
              "いいですね。そこへは京都駅からどうやって行けばいいですか？",
              "ありがとう。最後に、お昼におすすめの京都の食べ物を一つ教えてください。"]
user_audio = [tts(t) for t in user_texts]
print([f"{len(a) / SR:.1f} 秒" for a in user_audio])
display(Audio(user_audio[0], rate=SR))

['7.0 秒', '4.6 秒', '6.4 秒']


In [3]:
import re
import time
from threading import Thread

import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, TextIteratorStreamer, pipeline

BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo", dtype=DTYPE, device_map="auto")   # MIT

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto").eval()

def transcribe(audio):
    return asr(audio, generate_kwargs={"language": "ja", "task": "transcribe"})["text"].strip()

class StopAfterSentences(StoppingCriteria):
    """読み上げる応答は短くてよいので、n 文を生成したところで止める（3B のモデルは「1〜2 文で」の指示を守らないことが多い）"""
    def __init__(self, start, n=2):
        self.start, self.n = start, n
    def __call__(self, input_ids, scores, **kwargs):
        return len(re.findall(r"[。？]", tokenizer.decode(input_ids[0, self.start:]))) >= self.n     # 「もちろんです！」は数えない

def generate(messages, max_new_tokens=100):
    """応答を逐次受け取り、最初の文ができた時刻と、生成が終わった時刻を記録する"""
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    t0 = time.perf_counter()
    Thread(target=llm.generate, kwargs=dict(**inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer,
                                            stopping_criteria=[StopAfterSentences(inputs["input_ids"].shape[1])],
                                            pad_token_id=tokenizer.eos_token_id)).start()
    text, t_first = "", None
    for chunk in streamer:
        text += chunk
        if t_first is None and re.search(r"[。！？]", text):
            t_first = time.perf_counter() - t0
    t_all = time.perf_counter() - t0
    return text.strip(), t_first or t_all, t_all

# 1 回目は初期化で遅いので、測る前に 1 回ずつ動かしておく
transcribe(user_audio[0]); generate([{"role": "user", "content": "こんにちは"}], max_new_tokens=5); tts("こんにちは")
print("GPU メモリ:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB（Whisper と LLM）")

GPU メモリ: 8.3 GB（Whisper と LLM）


In [4]:
import pandas as pd

SYSTEM = ("あなたは旅行の相談に音声で答えるアシスタントです。答えはそのまま読み上げられるので、"
          "記号や箇条書きを使わず、1〜2 文の短い話し言葉で答えてください。場所を聞かれたら、具体的な名前を 1 つ挙げてください。")
history = [{"role": "system", "content": SYSTEM}]
rows, replies = [], []
for text, audio in zip(user_texts, user_audio):
    t = time.perf_counter(); heard = transcribe(audio); torch.cuda.synchronize(); t_asr = time.perf_counter() - t
    history.append({"role": "user", "content": heard})
    reply, t_llm_first, t_llm = generate(history)
    history.append({"role": "assistant", "content": reply})
    sentences = re.findall(r".+?[。！？]", reply) or [reply]
    t = time.perf_counter(); first_wav = tts(sentences[0]); t_tts_first = time.perf_counter() - t
    t = time.perf_counter(); wav = tts(reply); t_tts = time.perf_counter() - t
    replies.append(wav)
    print(f"利用者: {text}\n認識: {heard}\nボット: {reply}\n")
    rows.append({"発話の長さ": len(audio) / SR, "音声認識": t_asr, "LLM（最初の文）": t_llm_first, "LLM（全体）": t_llm,
                 "音声合成（最初の文）": t_tts_first, "音声合成（全体）": t_tts, "応答の長さ": len(wav) / SR})
df = pd.DataFrame(rows)
df["最初の音まで"] = df["音声認識"] + df["LLM（最初の文）"] + df["音声合成（最初の文）"]   # 文ごとに逐次処理した場合
df["全部そろうまで"] = df["音声認識"] + df["LLM（全体）"] + df["音声合成（全体）"]            # 段ごとに全部終えてから次へ渡す場合
display(df.round(2))
display(Audio(replies[0], rate=SR))

利用者: こんにちは。今度の週末に京都へ行くんですが、雨の日でも楽しめる場所はありますか？
認識: こんにちは。今度の週末に京都へ行くんですが、雨の日でも楽しめる場所はありますか?
ボット: もちろんです！京都には雨の日でも楽しめる場所がたくさんあります。例えば、清水寺や八坂神社などの寺社仏閣は、建物内を見学できるので雨でも大丈夫ですよ。

利用者: いいですね。そこへは京都駅からどうやって行けばいいですか？
認識: いいですね。そこへは京都駅からどうやって行けばいいですか?
ボット: 京都駅から清水寺へは、市バスを利用するのが便利です。具体的には、京都駅から「206系統」または「100系統」に乗って、「清水道」または「五条坂」で下車してください。

利用者: ありがとう。最後に、お昼におすすめの京都の食べ物を一つ教えてください。
認識: ありがとう。最後に、お昼におすすめの京都の食べ物を一つ教えてください。
ボット: 京都でお昼におすすめなのは、湯豆腐です。清水寺周辺にも湯豆腐のお店がたくさんありますので、ぜひお試しください。



,発話の長さ,音声認識,LLM（最初の文）,LLM（全体）,音声合成（最初の文）,音声合成（全体）,応答の長さ,最初の音まで,全部そろうまで
0,6.98,0.23,0.11,0.71,0.08,0.62,13.58,0.41,1.56
1,4.64,0.18,0.33,0.97,0.25,0.80,17.48,0.76,1.95
2,6.38,0.19,0.26,0.55,0.19,0.45,9.66,0.64,1.19


- 音声対話は、音声認識 → 対話 (LLM) → 音声合成の 3 段をつないだカスケード型で作るのが基本。各段を別々に取り替えられ、途中の書き起こしや応答の文字列を確かめられる。音声を直接入出力する end-to-end のモデルもあるが、日本語で T4 に載るものはまだ少ない。
- 検証の GPU では、5〜7 秒の発話の認識が約 0.2 秒、2 文の応答の生成が 0.5〜1.0 秒、10〜17 秒の応答の音声合成 (CPU) が 0.5〜0.8 秒で、全部そろうまで 1.2〜2.0 秒かかった。長い応答では生成と合成が同じくらいの時間を占める。T4 では GPU の段が数倍遅くなる。
- 人の会話では、相手が話し終えてから返事が始まるまでの間はふつう 1 秒もない。応答を全部作ってから読み上げるのではなく、LLM が 1 文を出すたびに合成して再生を始めると、最初の音が出るまでは 0.4〜0.8 秒になった（表の「最初の音まで」）。残りの文は、最初の文を再生している間に作ればよい。
- 書き言葉の長い応答は、聞くには長すぎる。最初の応答は読み上げると 13 秒を超えた。システムプロンプトで「1〜2 文の話し言葉で」「記号や箇条書きを使わない」と指示し、それでも長くなるので、2 文を生成したところで止めている (`StopAfterSentences`)。数字や英字、記号は読み上げで誤りやすいので、合成の前に読みを確かめるとよい（日本語の形態素解析と文分割の章の「発音・モーラ数・アクセントを取り出し、5-7-5 や読みの違いを調べる (pyopenjtalk)」）。
- 入力も同じ合成音声なので、認識はほぼ完全だった（句読点の全角・半角だけが違う）。実際の声、雑音、言いよどみのある音声では誤りが増え、その誤りが応答に響く。モデルによる精度と速さの違いは、データの収集と入出力の章の「Whisper で日本語の音声を書き起こす（モデルの比較、長い音声、字幕）」を参照。
- 応答の中身は、この章のほかのレシピと同じく 3B のモデルが知識から作ったもので、誤りを含みうる。この例では雨の日の行き先に、屋外を歩く清水寺を挙げた。
- Colab でマイクから録音するには、ブラウザの JavaScript で録音して Python に渡す必要がある（`google.colab` の `eval_js` を使う例が多い）。ブラウザの画面で話しかけるなら、この章の「ブラウザで使えるチャット画面を作る (Gradio)」の `gr.Audio(sources=["microphone"])` が手軽。話し終わりの検出 (VAD) や、ボットの発話中に割り込まれたときの処理も、実用では必要になる。
- Open JTalk の音声合成は CPU だけで速く動くが、声は機械的で、声質も 1 種類（同梱の HTS の音声 mei）。より自然な声には、ニューラルの音声合成（VITS 系など）を使う。合成音声を公開するときは、音声モデルの利用条件を確かめる。pyopenjtalk は MIT ライセンス、同梱の音声は CC BY 3.0（名古屋工業大学）、Whisper large-v3-turbo は MIT、sarashina2.2 は MIT ライセンス。